# 02 — Обучение модели
> **v2 | Агент удержания полосы + ПИД-регулятор | ветка: `v2-agent-pid`**

> 🕐 Обновлён: 2026-04-10 00:05 МСК

**Время:** ~10–15 минут на T4  
Загружаем из кэша `/content/data/cache.npy` (мгновенно), обучаем LaneCNN,  
сохраняем `/content/models/model_v2.pth` + копия на Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# Явно переходим в папку проекта
REPO_PATH = '/content/drive/MyDrive/diploma'
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

# Создаём нужные папки на Drive если их нет
os.makedirs('outputs', exist_ok=True)

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.dataset_v2 import load_and_cache, get_loaders
from src.model_v2   import build_model
from src.train_v2   import train, evaluate

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', DEVICE)

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Загружаем кэш — если нет, пересоздаём из датасета на Drive
import glob, pathlib, pandas as pd

CACHE_PATH = '/content/data/cache.npy'
LOCAL_DATA  = '/content/data'

if not pathlib.Path(CACHE_PATH).exists():
    print("Кэш не найден — распаковываю датасет и пересоздаю...")
    import zipfile, os

    ZIP_PATH = '/content/drive/MyDrive/diploma/data/dataset.zip'
    os.makedirs(LOCAL_DATA, exist_ok=True)

    if not pathlib.Path(f'{LOCAL_DATA}/IMG').exists():
        # Ищем IMG в уже распакованных подпапках
        img_dirs = glob.glob(f'{LOCAL_DATA}/**/IMG', recursive=True)
        if not img_dirs:
            print("Распаковываю zip...")
            with zipfile.ZipFile(ZIP_PATH, 'r') as z:
                z.extractall(LOCAL_DATA)
            print("Готово")
        else:
            print(f"IMG уже есть: {img_dirs[0]}")

    # Ищем CSV
    csvs = glob.glob(f'{LOCAL_DATA}/**/*.csv', recursive=True)
    if not csvs:
        raise FileNotFoundError("CSV не найден в /content/data/ — проверь распаковку")
    CSV_PATH = csvs[0]
    print(f"CSV: {CSV_PATH}")

    # Патчим Windows-пути
    _df = pd.read_csv(CSV_PATH)
    for _col in _df.columns[:3]:
        if _df[_col].dtype == object:
            _sample = str(_df[_col].iloc[0]).strip()
            if '\\' in _sample:
                _df[_col] = _df[_col].apply(
                    lambda x: 'IMG/' + str(x).strip().replace('\\', '/').split('/')[-1]
                    if pd.notna(x) else x
                )
                _fixed = str(pathlib.Path(CSV_PATH).parent) + '/driving_log_fixed.csv'
                _df.to_csv(_fixed, index=False)
                CSV_PATH = _fixed
                print(f"Пути исправлены → {CSV_PATH}")
                break

    IMAGES_DIR = str(pathlib.Path(CSV_PATH).parent) + '/'
    images, angles = load_and_cache(CSV_PATH, IMAGES_DIR, CACHE_PATH)
else:
    images, angles = load_and_cache(None, None, CACHE_PATH)

print(f'Загружено: {len(images)} сэмплов')

In [ ]:
# DataLoader: batch_size=128, num_workers=2, pin_memory=True
train_loader, val_loader, test_loader = get_loaders(
    images, angles, batch_size=128
)
print(f'Train батчей: {len(train_loader)}  '
      f'Val: {len(val_loader)}  Test: {len(test_loader)}')

In [ ]:
# Строим модель
model = build_model(DEVICE)

In [ ]:
# Патч train_v2.py — убираем verbose (удалён в PyTorch 2.2+)
import re
_path = '/content/drive/MyDrive/diploma/src/train_v2.py'
_txt  = open(_path).read()
_fixed = _txt.replace(
    'ReduceLROnPlateau(optimizer, factor=0.5, patience=3, verbose=True)',
    'ReduceLROnPlateau(optimizer, factor=0.5, patience=3)'
)
if _fixed != _txt:
    open(_path, 'w').write(_fixed)
    print('train_v2.py исправлен')
else:
    print('train_v2.py уже актуален')

# Перезагружаем модуль чтобы подхватить изменение
import importlib, src.train_v2
importlib.reload(src.train_v2)
from src.train_v2 import train, evaluate

In [ ]:
# Обучаем (20 эпох, EarlyStopping patience=5)
# Модель сохраняется локально — быстро
import os
os.makedirs('/content/models', exist_ok=True)

history = train(
    model, train_loader, val_loader,
    epochs=20,
    lr=3e-4,
    patience=5,
    model_path='/content/models/model_v2.pth',
    device=DEVICE
)

In [ ]:
# ── Кривые обучения ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='Train loss')
ax.plot(history['val_loss'],   label='Val loss')
ax.set_xlabel('Эпоха')
ax.set_ylabel('MSE Loss')
ax.set_title('Кривые обучения LaneCNN')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=120)
plt.show()

In [ ]:
# ── Метрики на тестовой выборке ───────────────────────────────────
model.load_state_dict(torch.load('/content/models/model_v2.pth', map_location=DEVICE))
metrics = evaluate(model, test_loader, DEVICE)

print('\n── Метрики на тесте ──────────────────')
print(f'  MSE:      {metrics["mse"]:.5f}')
print(f'  MAE:      {metrics["mae"]:.5f}')
print(f'  RMSE:     {metrics["rmse"]:.5f}')
print(f'  Accuracy: {metrics["accuracy"]*100:.1f}%  (в/вне полосы)')

# Копируем модель на Drive для сохранности
import shutil
os.makedirs('models', exist_ok=True)
shutil.copy('/content/models/model_v2.pth', 'models/model_v2.pth')
print('Модель скопирована на Drive: models/model_v2.pth')

In [ ]:
# ── Scatter: предсказание vs реальное значение ────────────────────
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for imgs, ang in test_loader:
        pred = model(imgs.to(DEVICE)).cpu().numpy().flatten()
        all_pred.extend(pred)
        all_true.extend(ang.numpy().flatten())

all_pred = np.array(all_pred)
all_true = np.array(all_true)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(all_true, all_pred, alpha=0.3, s=10)
ax.plot([-1, 1], [-1, 1], 'r--', linewidth=1.5, label='Идеал')
ax.axvline(-0.15, color='gray', linestyle=':', alpha=0.7)
ax.axvline( 0.15, color='gray', linestyle=':', alpha=0.7, label='Граница ±0.15')
ax.set_xlabel('Реальный угол')
ax.set_ylabel('Предсказанный угол')
ax.set_title('Предсказание vs Реальное')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/scatter.png', dpi=120)
plt.show()

In [ ]:
# -- Сравнение оптимизаторов: Adam (baseline) vs SGD vs RMSprop vs Adagrad --
# Обновлён: 2026-04-28 МСК
import torch.optim as optim

COMPARE_EPOCHS = 10
LR = 3e-4
SEED = 42

optimizers_cfg = [
    ('Adam (baseline)', lambda p: optim.Adam(p, lr=LR, weight_decay=1e-4)),
    ('SGD + momentum',  lambda p: optim.SGD(p, lr=LR, momentum=0.9, weight_decay=1e-4)),
    ('RMSprop',         lambda p: optim.RMSprop(p, lr=LR, weight_decay=1e-4)),
    ('Adagrad',         lambda p: optim.Adagrad(p, lr=LR, weight_decay=1e-4)),
]

results_opt = {}

for opt_name, opt_fn in optimizers_cfg:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    m = build_model(DEVICE)
    optimizer_i = opt_fn(m.parameters())
    criterion_i = torch.nn.MSELoss()
    train_losses_i, val_losses_i = [], []

    for epoch in range(COMPARE_EPOCHS):
        m.train()
        tl = 0.0
        for imgs, ang in train_loader:
            imgs, ang = imgs.to(DEVICE), ang.to(DEVICE)
            optimizer_i.zero_grad()
            loss = criterion_i(m(imgs), ang)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            optimizer_i.step()
            tl += loss.item() * len(imgs)
        train_losses_i.append(tl / len(train_loader.dataset))

        m.train(False)
        vl = 0.0
        with torch.no_grad():
            for imgs, ang in val_loader:
                imgs, ang = imgs.to(DEVICE), ang.to(DEVICE)
                vl += criterion_i(m(imgs), ang).item() * len(imgs)
        val_losses_i.append(vl / len(val_loader.dataset))

    results_opt[opt_name] = {'train': train_losses_i, 'val': val_losses_i}
    print(f'{opt_name:<25} val_loss={val_losses_i[-1]:.5f}')

# График
colors = ['#1F4788', '#E53935', '#2E7D32', '#F57C00']
styles = [('-', 2.5), ('--', 1.5), ('--', 1.5), ('--', 1.5)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for (name, res), color, (ls, lw) in zip(results_opt.items(), colors, styles):
    axes[0].plot(res['train'], color=color, lw=lw, ls=ls, label=name)
    axes[1].plot(res['val'],   color=color, lw=lw, ls=ls, label=name)

for ax, title in zip(axes, ['Train Loss (MSE)', 'Val Loss (MSE)']):
    ax.set_xlabel('Эпоха')
    ax.set_ylabel('MSE Loss')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle(f'Сравнение оптимизаторов ({COMPARE_EPOCHS} эпох, lr={LR}, Adam = baseline)', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/optimizer_comparison.png', dpi=120)
plt.show()

print('\n-- Итоговые метрики --')
print(f'{"Оптимизатор":<25} {"train_loss":>12} {"val_loss":>12}')
print('-' * 52)
for name, res in results_opt.items():
    marker = ' <- baseline' if 'baseline' in name else ''
    print(f'{name:<25} {res["train"][-1]:>12.5f} {res["val"][-1]:>12.5f}{marker}')
print('Сохранено: outputs/optimizer_comparison.png')

In [ ]:
# -- Эксперимент: CrossEntropy (BCELoss) vs MSE ---------------------
# Обновлён: 2026-04-29 МСК
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Бинарный классификатор (Sigmoid вместо Tanh) — только для сравнения
class LaneCNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, stride=2, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        with torch.no_grad():
            self._flat = int(self.features(torch.zeros(1,1,32,100)).numel())
        self.classifier = nn.Sequential(
            nn.Linear(self._flat, 64), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(64, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))

BCE_EPOCHS = 10
THRESHOLD  = 0.15

def angles_to_bce_loader(loader):
    all_imgs, all_labels = [], []
    for imgs, ang in loader:
        all_imgs.append(imgs)
        all_labels.append((ang.abs() < THRESHOLD).float())
    ds = TensorDataset(torch.cat(all_imgs), torch.cat(all_labels))
    return DataLoader(ds, batch_size=128, shuffle=False)

bce_train_dl = angles_to_bce_loader(train_loader)
bce_val_dl   = angles_to_bce_loader(val_loader)
bce_test_dl  = angles_to_bce_loader(test_loader)

torch.manual_seed(SEED)
clf           = LaneCNNClassifier().to(DEVICE)
bce_optim     = torch.optim.Adam(clf.parameters(), lr=LR, weight_decay=1e-4)
bce_criterion = torch.nn.BCELoss()

bce_val_losses = []
for epoch in range(BCE_EPOCHS):
    clf.train()
    for imgs, lbl in bce_train_dl:
        imgs, lbl = imgs.to(DEVICE), lbl.to(DEVICE)
        bce_optim.zero_grad()
        bce_criterion(clf(imgs), lbl).backward()
        torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
        bce_optim.step()
    clf.train(False)
    vl = 0.0
    with torch.no_grad():
        for imgs, lbl in bce_val_dl:
            imgs, lbl = imgs.to(DEVICE), lbl.to(DEVICE)
            vl += bce_criterion(clf(imgs), lbl).item() * len(imgs)
    bce_val_losses.append(vl / len(bce_val_dl.dataset))
    print(f'BCE epoch {epoch+1:02d}: val={bce_val_losses[-1]:.5f}')

# Точность на тесте
clf.train(False)
bce_correct = bce_total = 0
with torch.no_grad():
    for imgs, lbl in bce_test_dl:
        imgs, lbl = imgs.to(DEVICE), lbl.to(DEVICE)
        bce_correct += ((clf(imgs) >= 0.5).float() == lbl).sum().item()
        bce_total   += len(lbl)
bce_accuracy  = bce_correct / bce_total
mse_val_final = results_opt['Adam (baseline)']['val'][-1]

# График
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(results_opt['Adam (baseline)']['val'], 'b-', lw=2.5, label='Adam + MSELoss (baseline)')
ax.plot(bce_val_losses, 'r--', lw=2.0, label='Adam + BCELoss')
ax.set_xlabel('Эпоха'); ax.set_ylabel('Loss')
ax.set_title('CrossEntropy (BCE) vs MSE: val loss (10 эпох)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/crossentropy_vs_mse.png', dpi=120)
plt.show()

print('
-- MSE vs BCE --')
print(f'{"Метрика":<22} {"MSELoss (Tanh)":>18} {"BCELoss (Sigmoid)":>18}')
print('-' * 60)
print(f'{"Val Loss (10 эп.)":<22} {mse_val_final:>18.5f} {bce_val_losses[-1]:>18.5f}')
print(f'{"Accuracy (тест)":<22} {"N/A":>18} {bce_accuracy*100:>17.1f}%')
print('BCE-модель не идёт в агент — только сравнение.')
print('Сохранено: outputs/crossentropy_vs_mse.png')